# Bet 1 — RAMP + pull-push (Colab session 2)

RAMP backbone (recipe/seed 0) + **α·scaffold + β·glue** (anchor=clean stop-grad, positives = RAMP's own ℓ∞ & ℓ₁ adversarials, negatives = other-class adv, τ=0.1, α=β=0.5). Head discarded at eval; backbone saved as a RAMP-family state_dict → audited on the PC (12-AA).

## How to run — uploads (same as M0/M1b)
1. Upload to Drive `MyDrive/attackdro/`: **`cifar-10-python.tar.gz`** (the repo's, sha256 `6d958be0…`) and **`attackdro_code.zip`** (has `claim_b_rep.py` + `patch_ramp_claimb.py`).
2. Runtime → GPU. Edit the 3 paths. **Run all.** It clones RAMP, reuses YOUR uploaded CIFAR (no re-download), applies the patch, 1-epoch smoke, then full B1 (80ep).
3. Download `val_best.pth` from Drive → send to the PC for the 12-AA audit + B1−R.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')


## EDIT ME


In [ ]:
DRIVE_CIFAR_TARGZ = '/content/drive/MyDrive/attackdro/cifar-10-python.tar.gz'  # your uploaded CIFAR (reused, NOT re-downloaded)
DRIVE_CODE_ZIP    = '/content/drive/MyDrive/attackdro/attackdro_code.zip'      # has claim_b_rep.py + patch
DRIVE_OUT         = '/content/drive/MyDrive/attackdro/Bet1_out'                # val_best + logs land here
LBD = 5   # RAMP KL weight — MATCH the config that produced your R bar (armA_rampfull).
import os; os.makedirs(DRIVE_OUT, exist_ok=True)
assert os.path.exists(DRIVE_CIFAR_TARGZ), 'upload cifar-10-python.tar.gz to Drive first'
assert os.path.exists(DRIVE_CODE_ZIP),   'upload attackdro_code.zip to Drive first'
print('paths OK')


In [ ]:
# Clone RAMP (public) + deps. RAMP needs robustbench (for the Wong2020Fast init used by the RAMP recipe).
![ -d /content/RAMP ] || git clone -q https://github.com/uiuc-focal-lab/RAMP /content/RAMP
!pip -q install robustbench wandb 2>&1 | tail -2
import importlib.util; print('robustbench importable:', importlib.util.find_spec('robustbench') is not None)


In [ ]:
# ===DATA FIX=== reuse YOUR uploaded CIFAR tarball (hash-checked) -> /content/data (RAMP's --data_dir finds it, no download)
import hashlib, tarfile, os
h = hashlib.sha256(open(DRIVE_CIFAR_TARGZ,'rb').read()).hexdigest()
print('cifar tar sha256:', h[:16], '(PC =', '6d958be074577803', ')')
assert h.startswith('6d958be074577803'), 'CIFAR tar mismatch — upload the repo data/cifar-10-python.tar.gz'
os.makedirs('/content/data', exist_ok=True)
with tarfile.open(DRIVE_CIFAR_TARGZ) as t: t.extractall('/content/data')   # -> /content/data/cifar-10-batches-py
assert os.path.exists('/content/data/cifar-10-batches-py/test_batch'), 'extract failed'
print('CIFAR ready at /content/data (same bytes as the PC / M1a)')


In [ ]:
# Get our code from the zip; put claim_b_rep.py + patch where RAMP_claimB imports them; set CLAIMB_REP_DIR
import zipfile, shutil, os
os.makedirs('/content/attackdro', exist_ok=True)
with zipfile.ZipFile(DRIVE_CODE_ZIP) as z: z.extractall('/content/attackdro')
shutil.copy('/content/attackdro/scripts/dev/claim_b_rep.py', '/content/RAMP/claim_b_rep.py')
shutil.copy('/content/attackdro/scripts/dev/patch_ramp_claimb.py', '/content/RAMP/patch_ramp_claimb.py')
os.environ['CLAIMB_REP_DIR'] = '/content/RAMP'
print('claim_b_rep + patch in /content/RAMP')


In [ ]:
# Apply the patch -> RAMP_claimB.py ; verify syntax
%cd /content/RAMP
!python patch_ramp_claimb.py
import ast; ast.parse(open('/content/RAMP/RAMP_claimB.py').read()); print('RAMP_claimB.py SYNTAX OK')


## SMOKE — 1 epoch B1 (validate on Colab before the 80-ep run)


In [ ]:
AT,EP,SD = 2,1,'/content/smoke'
%cd /content/RAMP
import os
cmd = (f"CLAIMB_REP_DIR=/content/RAMP python RAMP_claimB.py --lr-max 0.05 --lr-schedule=static "
       f"--at_iter {AT} --epochs {EP} --save_freq 1 --eval_freq 1 --fname B1_smoke "
       f"--kl --max --gp --lbd {LBD} --seed 0 --claimB B1 --cb_alpha 0.5 --cb_beta 0.5 "
       f"--data_dir /content/data --save_dir '{SD}'")
print(cmd); get_ipython().system(cmd)
print('SMOKE val_best:', os.path.exists('/content/smoke/B1_smoke/val_best.pth'))


## Full B1 (80 ep, seed 0) → Drive


In [ ]:
AT,EP,SD = 10,80,DRIVE_OUT
%cd /content/RAMP
import os
cmd = (f"CLAIMB_REP_DIR=/content/RAMP python RAMP_claimB.py --lr-max 0.05 --lr-schedule=static "
       f"--at_iter {AT} --epochs {EP} --save_freq 10 --eval_freq 10 --fname B1_pullpush_seed0 "
       f"--kl --max --gp --lbd {LBD} --seed 0 --claimB B1 --cb_alpha 0.5 --cb_beta 0.5 "
       f"--data_dir /content/data --save_dir '{SD}'")
print(cmd); get_ipython().system(cmd)


## Retrieve
- `Bet1_out/B1_pullpush_seed0/val_best.pth` (+ RAMP's `ep_*.pth`, logs) on Drive → send `val_best.pth` to the PC → 12-AA audit (`--model-family ramp`) + **B1 − R** paired bootstrap.
- ⚠ Config-match: `lbd`/flags must match the config that produced R (armA_rampfull) for a clean B1−R read.
